# Osnove strojnega učenja · Prvi koraki

**Stopnja** 1 · **Predviden čas** okoli 20-30 minut

>Spoznali bomo nekaj osnovnih pojmov strojnega učenja in začeli z najosnovnejšim vprašanjem: kaj sploh je strojno učenje in zakaj ga potrebujemo?
Na koncu te lekcije boš, upamo, v strojnem učenju videl orodje, ki nam na podlagi podatkov pomaga sprejemati boljše odločitve in napovedi.

Za učenje iz tega zvezka ga moraš le pognati. Kode ti ni treba spreminjati ali razumeti.

## Pomagaj zasedenemu študentu

Marc je študent arheologije. S profesorico sta na izkopališču odkrila na stotine črepinj (delčkov) keramike. Prva analiza je pokazala, da črepinje pripadajo dvema starorimskima vazama, ki ju bomo imenovali vaza A in vaza B. Profesorica želi vedeti, katera črepinja pripada kateri vazi, da bi vazi lahko sestavila, a sama za to nima časa. Vsako črepinjo je treba pregledati pod mikroskopom, da ugotovimo, kateri vazi pripada. Zato to dolgotrajno in dolgočasno delo preda Marcu. Danes je petek, profesorica pa želi imeti črepinje razvrščene do ponedeljka, da bo lahko začela sestavljati vazi.

A Marc nima časa. Za vikend ima že načrte, saj gre s prijatelji na glasbeni festival. Da bi črepinje hitro razvrstil, jih sam razvrsti le 100, preostalih 400 pa namerava razvrstiti z modelom strojnega učenja. Predvideva, da je lega črepinje na izkopališču dober pokazatelj, kateri vazi pripada, saj so črepinje iste vaze verjetno blizu skupaj. Zato si za vsako črepinjo zapiše tudi njene koordinate.

Težava je, da je Marc študent arheologije in o strojnem učenju ne ve ničesar, zato prosi tebe za pomoč. Pomagaj mu, kolikor najbolje znaš: bolj ko se boš motil, v večjih težavah bo Marc pri profesorici.

In [1]:
#@title Poženi celico (klikni ▶ na levi ali Shift+Enter). Naložili se bodo podatki in orodja. Kode ni potrebno brati.
# Section for the imports. Imports are libraries that contain useful functions and classes that we can use in our code.
import io
import json
import uuid

import ipywidgets as widgets
import matplotlib.pyplot as plt     # Matplotlib library for plotting
import numpy as np                  # Numpy library for numerical computing
from IPython.display import HTML, display
from sklearn.datasets import make_moons     # Scikit-learn library is the major machine learning library in Python. You will use it a lot.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 42
CLASS_COLORS = ["#386cb0", "#f0027f"]
classes = ["Vaza A", "Vaza B"]      # Fragment classes. Class 0 is Vase A and class 1 is Vase B.

# make_moons generates the data for you. X is a 2D array of shape (100, 2) containing the coordinates of the fragments,
# and y is a 1D array of shape (100,) containing the labels (0 or 1) for each fragment.
X, y = make_moons(n_samples=100, noise=0.3, random_state=SEED)

# The 400 fragments Marc did not sort himself, with their true labels. We only look at them at the very end.
X_new, y_new = make_moons(n_samples=400, noise=0.3, random_state=SEED + 1)

PAD = 0.35  # how far the plots reach beyond the outermost fragment
XLIM = (X[:, 0].min() - PAD, X[:, 0].max() + PAD)
YLIM = (X[:, 1].min() - PAD, X[:, 1].max() + PAD)


# --------------------------------------------------
# Plots drawn with matplotlib
# --------------------------------------------------

def plot_dataset(X, y, ax, marker="o", alpha=0.8, classes=classes):
    """Draw the fragments as dots, coloured by the vase they belong to."""
    for label in (0, 1):
        ax.scatter(X[y == label, 0], X[y == label, 1], c=CLASS_COLORS[label],
                   label=classes[label], alpha=alpha, marker=marker)
    ax.set(xlabel="x", ylabel="y", xlim=XLIM, ylim=YLIM)
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.4)
    return ax


def plot_decision_boundary(model, ax):
    """Colour every place in the dig site by the vase the model predicts there."""
    # First we construct a grid of points ...
    xx, yy = np.meshgrid(np.linspace(*XLIM, 250), np.linspace(*YLIM, 250))
    # ... then we ask the model which vase each point belongs to ...
    prediction = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    # ... and colour the points by the predicted vase.
    ax.contourf(xx, yy, prediction, levels=[-.5, .5, 1.5], colors=CLASS_COLORS, alpha=.18)


def show_dataset():
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_dataset(X, y, ax)
    ax.set_title("Lega črepinj na izkopališču in vaza, ki ji pripadajo")
    plt.show()


def _png(fig):
    """The figure as PNG bytes, for an image widget. Drawing into an image widget works the same in Colab, VS Code and Jupyter."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buffer.getvalue()


def _picture_widget():
    return widgets.Image(format="png", layout=widgets.Layout(width="680px", max_width="100%", height="auto"))


# --------------------------------------------------
# Models
# --------------------------------------------------

def polynomial_logistic_regression(degree):
    """Logistic regression whose boundary is a polynomial of the given degree."""
    return make_pipeline(
        PolynomialFeatures(degree=degree),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED),
    )


def evaluate_model(model):
    """Fit the model on the 100 sorted fragments and return its accuracy on them."""
    model.fit(X, y)                     # This fits the model to the data: the computer learns the best boundary.
    predictions = model.predict(X)      # This makes predictions on the data using the fitted model.
    return accuracy_score(y, predictions)


def logistic_regression_degree():
    """A slider for the degree of the polynomial, and the boundary logistic regression learns with it."""
    degree = widgets.IntSlider(value=1, min=1, max=12, description="Stopnja:", continuous_update=False)
    score = widgets.HTML()
    picture = _picture_widget()

    def update(change=None):
        model = polynomial_logistic_regression(degree.value)
        accuracy = evaluate_model(model)
        fig, ax = plt.subplots(figsize=(7, 5))
        plot_decision_boundary(model, ax)
        plot_dataset(X, y, ax)
        ax.set_title(f"Stopnja polinoma {degree.value}")
        picture.value = _png(fig)
        score.value = f"<b>Točnost: {accuracy:.1%}</b>"

    degree.observe(update, names="value")
    update()
    display(widgets.VBox([degree, score, picture]))


MODEL_NAMES = ["Logistična regresija", "K najbližjih sosedov", "Odločitveno drevo", "SVM (RBF)", "Nevronska mreža"]


def model_playground(new_data=False):
    """Choose a model and its settings, and see the boundary it learns from the 100 sorted fragments.
    With new_data=True, the other 400 fragments are drawn too, and the model is scored on them as well."""
    choice = widgets.Dropdown(options=MODEL_NAMES, description="Model:")
    slider = dict(continuous_update=False, style={"description_width": "110px"})
    degree = widgets.IntSlider(value=3, min=1, max=12, description="stopnja", **slider)
    neighbors = widgets.IntSlider(value=5, min=1, max=30, description="sosedi", **slider)
    depth = widgets.IntSlider(value=5, min=1, max=15, description="največja globina", **slider)
    svm_c = widgets.FloatLogSlider(value=1, base=10, min=-2, max=3, step=.1, description="C", **slider)
    gamma = widgets.FloatLogSlider(value=.1, base=10, min=-3, max=1, step=.1, description="gamma", **slider)
    layers = widgets.IntSlider(value=2, min=1, max=3, description="skrite plasti", **slider)
    neurons = widgets.IntSlider(value=20, min=5, max=60, step=5, description="nevroni/plast", **slider)
    settings = {
        "Logistična regresija": [degree],
        "K najbližjih sosedov": [neighbors],
        "Odločitveno drevo": [depth],
        "SVM (RBF)": [svm_c, gamma],
        "Nevronska mreža": [layers, neurons],
    }
    controls = [degree, neighbors, depth, svm_c, gamma, layers, neurons]
    score = widgets.HTML()
    picture = _picture_widget()

    def selected_model():
        name = choice.value
        if name == "Logistična regresija":
            return polynomial_logistic_regression(degree.value)
        if name == "K najbližjih sosedov":
            return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=neighbors.value))
        if name == "Odločitveno drevo":
            return DecisionTreeClassifier(max_depth=depth.value, random_state=SEED)
        if name == "SVM (RBF)":
            return make_pipeline(StandardScaler(), SVC(C=svm_c.value, gamma=gamma.value))
        hidden = tuple([neurons.value] * layers.value)
        return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=hidden, max_iter=2000, random_state=SEED))

    def update(change=None):
        for control in controls:
            control.layout.display = "" if control in settings[choice.value] else "none"
        model = selected_model()
        accuracy = evaluate_model(model)
        fig, ax = plt.subplots(figsize=(7, 5))
        plot_decision_boundary(model, ax)
        if new_data:
            plot_dataset(X, y, ax, alpha=0.5, classes=["Prvih 100: vaza A", "Prvih 100: vaza B"])
            plot_dataset(X_new, y_new, ax, marker="x", alpha=0.35, classes=["Novih 400: vaza A", "Novih 400: vaza B"])
            new_accuracy = accuracy_score(y_new, model.predict(X_new))
            score.value = (f"<b>Točnost na prvih 100 črepinjah: {accuracy:.1%}</b><br>"
                           f"<b>Točnost na novih 400 črepinjah: {new_accuracy:.1%}</b>")
        else:
            plot_dataset(X, y, ax)
            score.value = f"<b>Točnost: {accuracy:.1%}</b>"
        ax.set_title(choice.value)
        picture.value = _png(fig)

    for control in [choice, *controls]:
        control.observe(update, names="value")
    update()
    display(widgets.VBox([choice, *controls, score, picture]))


# --------------------------------------------------
# Plots drawn in the browser
# --------------------------------------------------
# The two pages below run in the browser, so the plot answers the mouse and the sliders at once, without Python.
# Both share the plotting code in _PLOT_JS. The fragments come in as JSON.

_PLOT_JS = """
function makePlot(root, data) {
  const W = 640, H = 470, M = {l: 48, r: 14, t: 14, b: 42};
  const canvas = root.querySelector("canvas"), ctx = canvas.getContext("2d");
  const dpr = window.devicePixelRatio || 1;
  canvas.width = W * dpr;
  canvas.height = H * dpr;
  ctx.scale(dpr, dpr);
  const [x0, x1] = data.xlim, [y0, y1] = data.ylim;
  const sx = x => M.l + (x - x0) / (x1 - x0) * (W - M.l - M.r);
  const sy = v => H - M.b - (v - y0) / (y1 - y0) * (H - M.t - M.b);

  // A grid of points over the whole plot; each one is coloured by the vase predicted there.
  const NX = 128, NY = 94, grid = [];
  for (let j = 0; j < NY; j++) for (let i = 0; i < NX; i++)
    grid.push([x0 + (i + 0.5) / NX * (x1 - x0), y0 + (j + 0.5) / NY * (y1 - y0)]);

  function ticks(lo, hi) {
    const raw = (hi - lo) / 6, mag = Math.pow(10, Math.floor(Math.log10(raw)));
    const step = [1, 2, 2.5, 5, 10].map(m => m * mag).find(s => s >= raw);
    const out = [];
    for (let v = Math.ceil(lo / step) * step; v <= hi + 1e-9; v += step) out.push(+v.toFixed(6));
    return out;
  }

  // Which side of the boundary a point is on: positive on one side, negative on the other.
  // Both ways of naming the sides are tried, and the one that gets more fragments right is kept.
  function classify(side) {
    const s = data.X.map(side);
    const correct = flip => s.filter((v, k) => (flip ? (v < 0 ? 1 : 0) : (v > 0 ? 1 : 0)) === data.y[k]).length;
    const flip = correct(true) > correct(false);
    const predict = v => flip ? (v < 0 ? 1 : 0) : (v > 0 ? 1 : 0);
    const mistakes = s.map((v, k) => predict(v) !== data.y[k]);
    const wrong = mistakes.filter(Boolean).length;
    return {mistakes, wrong, accuracy: 1 - wrong / data.y.length, regions: grid.map(p => predict(side(p)))};
  }

  function draw(state) {
    ctx.fillStyle = "#fff";
    ctx.fillRect(0, 0, W, H);
    ctx.save();
    ctx.beginPath();
    ctx.rect(M.l, M.t, W - M.l - M.r, H - M.t - M.b);
    ctx.clip();
    if (state.regions) {
      const cw = (W - M.l - M.r) / NX, ch = (H - M.t - M.b) / NY;
      ctx.globalAlpha = 0.18;
      state.regions.forEach((k, n) => {
        ctx.fillStyle = data.colors[k];
        ctx.fillRect(M.l + (n % NX) * cw, H - M.b - (Math.floor(n / NX) + 1) * ch, cw + 0.6, ch + 0.6);
      });
      ctx.globalAlpha = 1;
    }
    ctx.strokeStyle = "#ddd";
    ctx.lineWidth = 1;
    for (const v of ticks(x0, x1)) { ctx.beginPath(); ctx.moveTo(sx(v), M.t); ctx.lineTo(sx(v), H - M.b); ctx.stroke(); }
    for (const v of ticks(y0, y1)) { ctx.beginPath(); ctx.moveTo(M.l, sy(v)); ctx.lineTo(W - M.r, sy(v)); ctx.stroke(); }
    data.X.forEach(([a, b], k) => {
      ctx.globalAlpha = 0.8;
      ctx.fillStyle = data.colors[data.y[k]];
      ctx.beginPath(); ctx.arc(sx(a), sy(b), 4.5, 0, 2 * Math.PI); ctx.fill();
      ctx.globalAlpha = 1;
      if (state.mistakes && state.mistakes[k]) {
        ctx.strokeStyle = "#000"; ctx.lineWidth = 1.5;
        ctx.beginPath(); ctx.arc(sx(a), sy(b), 8, 0, 2 * Math.PI); ctx.stroke();
      }
    });
    if (state.line && state.line.length > 1) {
      ctx.strokeStyle = "#000"; ctx.lineWidth = 2.5; ctx.lineJoin = "round";
      ctx.beginPath();
      state.line.forEach(([a, b], k) => k ? ctx.lineTo(sx(a), sy(b)) : ctx.moveTo(sx(a), sy(b)));
      ctx.stroke();
    }
    ctx.restore();

    // Frame, tick labels and axis names.
    ctx.strokeStyle = "#000"; ctx.lineWidth = 1;
    ctx.strokeRect(M.l, M.t, W - M.l - M.r, H - M.t - M.b);
    ctx.fillStyle = "#000"; ctx.font = "12px sans-serif";
    ctx.textAlign = "center"; ctx.textBaseline = "top";
    for (const v of ticks(x0, x1)) ctx.fillText(v, sx(v), H - M.b + 5);
    ctx.fillText("x", (M.l + W - M.r) / 2, H - 18);
    ctx.textAlign = "right"; ctx.textBaseline = "middle";
    for (const v of ticks(y0, y1)) ctx.fillText(v, M.l - 6, sy(v));
    ctx.fillText("y", 14, (M.t + H - M.b) / 2);

    // Legend.
    const entries = [[0, data.classes[0]], [1, data.classes[1]]];
    if (state.mistakes) entries.push([-1, "Napake"]);
    const lx = W - M.r - 100, ly = M.t + 8;
    ctx.globalAlpha = 0.85; ctx.fillStyle = "#fff";
    ctx.fillRect(lx - 8, ly - 4, 102, entries.length * 18 + 6);
    ctx.globalAlpha = 1; ctx.strokeStyle = "#ccc"; ctx.strokeRect(lx - 8, ly - 4, 102, entries.length * 18 + 6);
    ctx.textAlign = "left";
    entries.forEach(([k, text], n) => {
      const cy = ly + 8 + n * 18;
      ctx.beginPath(); ctx.arc(lx + 4, cy, k < 0 ? 6 : 4.5, 0, 2 * Math.PI);
      if (k < 0) { ctx.strokeStyle = "#000"; ctx.lineWidth = 1.5; ctx.stroke(); }
      else { ctx.fillStyle = data.colors[k]; ctx.fill(); }
      ctx.fillStyle = "#000"; ctx.fillText(text, lx + 16, cy);
    });
  }

  // From a mouse or touch event to the place in the dig site under it.
  function toData(e) {
    const box = canvas.getBoundingClientRect();
    const px = (e.clientX - box.left) / box.width * W, py = (e.clientY - box.top) / box.height * H;
    return [x0 + (px - M.l) / (W - M.l - M.r) * (x1 - x0), y0 + (H - M.b - py) / (H - M.t - M.b) * (y1 - y0)];
  }

  return {draw, classify, toData, xlim: data.xlim, ylim: data.ylim, scale: (x1 - x0) / (W - M.l - M.r)};
}
"""

_STYLE = """
  <style>
    #__UID__ { background: #fff; color: #000; padding: 12px; font-family: sans-serif; border-radius: 6px; max-width: 664px; }
    #__UID__ .bar { display: flex; gap: 16px; align-items: center; flex-wrap: wrap; margin-bottom: 8px; font-size: 14px; }
    #__UID__ .bar label { display: flex; gap: 8px; align-items: center; }
    #__UID__ .bar input[type=range] { width: 220px; }
    #__UID__ .bar .value { display: inline-block; min-width: 36px; font-weight: bold; }
    #__UID__ button { font-size: 14px; padding: 4px 14px; border: 1px solid #d08000; background: #f0ad4e; color: #000;
                      border-radius: 4px; cursor: pointer; }
    #__UID__ .score { font-size: 15px; min-height: 2.6em; margin-bottom: 6px; }
    #__UID__ canvas { display: block; width: 640px; max-width: 100%; height: auto; touch-action: none; }
  </style>
"""

# The page draw_your_boundary shows: draw a curve with the mouse, and see how many fragments it puts on the wrong side.
_DRAW_HTML = """
<div id="__UID__">
__STYLE__
  <div class="bar"><button>Ponastavi</button></div>
  <div class="score">Nariši krivuljo in spusti miško, da jo oceniš.</div>
  <canvas style="cursor: crosshair"></canvas>
  <script>
  (() => {
    __PLOT_JS__
    const root = document.getElementById("__UID__");
    const plot = makePlot(root, __DATA__);
    const canvas = root.querySelector("canvas"), score = root.querySelector(".score");
    let line = [], drawing = false;

    // Which side of the drawn curve a point is on: the side of the piece of the curve closest to it.
    function side(p) {
      let best = Infinity, s = 0;
      for (let k = 0; k + 1 < line.length; k++) {
        const [ax, ay] = line[k], dx = line[k + 1][0] - ax, dy = line[k + 1][1] - ay, length = dx * dx + dy * dy;
        if (length < 1e-12) continue;
        const t = Math.max(0, Math.min(1, ((p[0] - ax) * dx + (p[1] - ay) * dy) / length));
        const cx = ax + t * dx, cy = ay + t * dy, d = (p[0] - cx) ** 2 + (p[1] - cy) ** 2;
        if (d < best) { best = d; s = dx * (p[1] - cy) - dy * (p[0] - cx); }
      }
      return s;
    }

    canvas.addEventListener("pointerdown", e => {
      canvas.setPointerCapture(e.pointerId);
      drawing = true;
      line = [plot.toData(e)];
      score.textContent = "Spusti miško, da oceniš svojo mejo.";
      plot.draw({line});
    });
    canvas.addEventListener("pointermove", e => {
      if (!drawing) return;
      const p = plot.toData(e), last = line[line.length - 1];
      if (Math.hypot(p[0] - last[0], p[1] - last[1]) < 3 * plot.scale) return;  // skip moves of under 3 pixels
      line.push(p);
      plot.draw({line});
    });
    const release = () => {
      if (!drawing) return;
      drawing = false;
      if (line.length < 3) {
        score.textContent = "Nariši daljšo krivuljo.";
        return;
      }
      const result = plot.classify(side);
      plot.draw({line, ...result});
      score.innerHTML = `<b>Točnost: ${(100 * result.accuracy).toFixed(1)}%</b> · ${result.wrong} napak`;
    };
    canvas.addEventListener("pointerup", release);
    canvas.addEventListener("pointercancel", release);
    root.querySelector("button").addEventListener("click", () => {
      line = [];
      score.textContent = "Nariši krivuljo in spusti miško, da jo oceniš.";
      plot.draw({});
    });
    plot.draw({});
  })();
  </script>
</div>
"""

# The page straight_line shows: two sliders set the slope and the intercept of a straight boundary.
_LINE_HTML = """
<div id="__UID__">
__STYLE__
  <div class="bar">
    <label>Naklon (m): <input type="range" class="slope" min="-5" max="5" step="0.1" value="0"><span class="value"></span></label>
    <label>Odsek (b): <input type="range" class="intercept" min="__YMIN__" max="__YMAX__" step="0.1" value="0"><span class="value"></span></label>
  </div>
  <div class="score"></div>
  <canvas></canvas>
  <script>
  (() => {
    __PLOT_JS__
    const root = document.getElementById("__UID__");
    const plot = makePlot(root, __DATA__);
    const slope = root.querySelector(".slope"), intercept = root.querySelector(".intercept");
    const score = root.querySelector(".score");

    function update() {
      const m = +slope.value, b = +intercept.value;
      slope.nextElementSibling.textContent = m.toFixed(1);
      intercept.nextElementSibling.textContent = b.toFixed(1);
      const result = plot.classify(p => p[1] - (m * p[0] + b));  // above the line or below it
      const line = plot.xlim.map(x => [x, m * x + b]);
      plot.draw({line, ...result});
      score.innerHTML = `<b>Točnost: ${(100 * result.accuracy).toFixed(1)}%</b> · ${result.wrong} napak` +
                        `<br><small>Odločitvena meja: y = ${m.toFixed(1)}x ${b < 0 ? "−" : "+"} ${Math.abs(b).toFixed(1)}</small>`;
    }
    slope.addEventListener("input", update);
    intercept.addEventListener("input", update);
    update();
  })();
  </script>
</div>
"""


def _page(template, **values):
    """Fill in a page template and show it."""
    data = {
        "X": np.round(X, 4).tolist(),
        "y": y.tolist(),
        "xlim": [float(v) for v in XLIM],
        "ylim": [float(v) for v in YLIM],
        "colors": CLASS_COLORS,
        "classes": classes,
    }
    page = template.replace("__STYLE__", _STYLE).replace("__PLOT_JS__", _PLOT_JS)
    for key, value in {"__UID__": f"ml-{uuid.uuid4().hex}", "__DATA__": json.dumps(data), **values}.items():
        page = page.replace(key, value)
    display(HTML(page))


def draw_your_boundary():
    """Draw a boundary with the mouse. On release, the accuracy of your boundary is shown and the mistakes are circled."""
    _page(_DRAW_HTML)


def straight_line():
    """A straight boundary, y = m·x + b. Two sliders set m and b."""
    _page(_LINE_HTML, __YMIN__=f"{np.floor(YLIM[0] * 10) / 10:.1f}", __YMAX__=f"{np.ceil(YLIM[1] * 10) / 10:.1f}")

Tukaj je 100 črepinj, ki jih je razvrstil Marc. Vsaka pika je ena črepinja na svojem mestu na izkopališču, obarvana glede na vazo, ki ji pripada.

In [2]:
show_dataset()

Preden uporabiš model, poskusi pravilo poiskati sam. Poženi celico, nato pa z miško klikni in povleci čez graf, da narišeš mejo. Nad grafom se bo izpisala točnost razvrstitve z tvojo mejo. S **Ponastavi** počistiš graf in poskusiš znova. Lahko pa preprosto narišeš novo mejo.

In [3]:
draw_your_boundary()

Risanje meje je dober način, da ločimo razreda, a je ročno delo. V tem ni nič »strojnega učenja«. Namesto da mejo rišemo sami, bi raje, da se računalnik sam nauči najboljše možne meje.

Najprej izberemo model, nato pa računalniku naročimo, naj se nauči najboljše meje za ta model. Nič hudega, če ne veš, kaj je model. Za zdaj si ga predstavljaj kot nabor pravil, s katerimi lahko računalnik loči razreda.

Prvi model, ki ga lahko preizkusimo, loči razreda z ravno črto. Pravili, ki ju lahko spreminjamo, sta torej naklon in odsek premice na osi y. Poženi spodnjo celico in poskusi najti premico, ki razreda loči najbolje. Naklon in odsek spreminjaš z drsnikoma.

In [4]:
straight_line()

Si zadovoljen z rezultatom? Upamo, da si dosegel dobro točnost. Model, ki si ga pravkar prilagodil, je primer **logistične regresije**. Kot vidiš, ravna črta ni dovolj, da bi razreda ločili. Na srečo lahko logistično regresijo razširimo, da razreda loči s kvadratnim, kubičnim ali celo polinomom višje stopnje. Poženi spodnjo celico in spreminjaj stopnjo polinoma, da vidiš, kako vpliva na točnost.

In [5]:
logistic_regression_degree()

Čestitke! Pravkar si naučil svoj prvi model strojnega učenja. S spreminjanjem stopnje polinoma si spreminjal zahtevnost modela. V ozadju se je računalnik sam naučil najboljših parametrov.

A logistična regresija ni edini model. V naslednjem delu lahko izbiraš med **k najbližjimi sosedi**, **odločitvenimi drevesi**, **metodo podpornih vektorjev (SVM)**, **nevronskimi mrežami** in seveda **logistično regresijo**. Vse te modele bomo spoznali med tečajem, za zdaj pa jih le preizkusi in poglej, kako se odrežejo. Zapiši si nastavitve, ki so pri vsakem modelu dale najboljšo točnost. Kasneje ti bodo prišle prav.

Kateri model in katere nastavitve bi na podlagi svojih poskusov priporočil Marcu za razvrščanje črepinj?

In [6]:
model_playground()

>Za radovedne: odločitvena drevesa rišejo meje, ki so povsem navpične ali vodoravne. Ali ugotoviš, zakaj?

### Po vikendu

Ponedeljek zjutraj je. Marc je po tvojem priporočilu z izbranim modelom in nastavitvami razvrstil preostalih 400 črepinj. Ne da bi poznal njihove prave oznake, je sestavil obe vazi. A ko je sestavljeni vazi pokazal profesorici, ta ni bila zadovoljna. Rekla je, da vazi nista pravilno sestavljeni in da je Marc gotovo napačno razvrstil nekaj črepinj.

Z Marcom sta morala nato vse črepinje ročno razvrstiti znova. Marc ti je kasneje dal preostalih 400 črepinj z njihovimi pravimi oznakami. Ponovno se poigraj z istimi modeli in nastavitvami kot prej in poglej, kako dobro se odrežejo na novih podatkih. Ali večja točnost na prvih 100 črepinjah pomeni tudi večjo točnost na novih 400 črepinjah?

In [7]:
model_playground(new_data=True)